# Chapter 49: Self-Supervised & Foundation Vision Models
## Multimedia & Computer Vision Learning Hub — ITERA
**Author:** Aji Pamungkas Tri Nurcahyo  
**Objective:** Menguasai prinsip pembelajaran representasi visual mandiri (Self-Supervised Learning), menganalisis formulasi matematis fungsi rugi InfoNCE, mengevaluasi pengaruh parameter temperatur $\tau$, serta mensimulasikan mekanisme klasifikasi zero-shot multimodal berbasis model fondasi CLIP.

### 1. Landasan Teori & Formulasi Rugi InfoNCE
Pembelajaran kontrastif memaksimalkan batas bawah mutual information melalui penataan ruang representasi di mana representasi kueri $\mathbf{q}$ ditarik mendekati pasangan positif $\mathbf{k}_+$ dan didorong menjauhi himpunan pasangan negatif $\mathcal{K}^-$.

Fungsi rugi InfoNCE dirumuskan sebagai:
$$\mathcal{L}_{\text{InfoNCE}} = -\log \frac{\exp\left( \frac{\mathbf{q} \cdot \mathbf{k}_+}{\tau} \right)}{\exp\left( \frac{\mathbf{q} \cdot \mathbf{k}_+}{\tau} \right) + \sum_{j=1}^K \exp\left( \frac{\mathbf{q} \cdot \mathbf{k}_j^-}{\tau} \right)}$$
Seluruh vektor representasi dinormalisasi $\ell_2$ sehingga dot product ekuivalen dengan kesamaan kosinus.

In [ ]:
# 2. Setup Lingkungan & Dual-Engine Check
import numpy as np
import sys

print(f"Python Version: {sys.version}")
print(f"NumPy Version: {np.__version__}")

HAS_TORCH = False
try:
    import torch
    import torch.nn.functional as F
    HAS_TORCH = True
    print(f"PyTorch: {torch.__version__} (Device: {'cuda' if torch.cuda.is_available() else 'cpu'})")
except ImportError:
    print("INFO: PyTorch tidak terpasang di host lokal. Beroperasi dalam Mode Analisis Algoritmik NumPy murni.")

In [ ]:
# 3. Implementasi InfoNCE & Validasi Analitik Terpandu
def l2_norm(x, axis=-1, eps=1e-12):
    norm = np.linalg.norm(x, axis=axis, keepdims=True)
    return x / np.maximum(norm, eps)

def infonce_loss(q, k_pos, k_negs, tau=0.1):
    q_u = l2_norm(q)
    k_pos_u = l2_norm(k_pos)
    k_negs_u = l2_norm(k_negs)
    
    s_pos = float(np.dot(q_u, k_pos_u))
    s_negs = np.dot(k_negs_u, q_u)
    
    logits = np.concatenate(([s_pos], s_negs)) / tau
    max_l = np.max(logits)
    exp_l = np.exp(logits - max_l)
    probs = exp_l / np.sum(exp_l)
    
    loss = -float(np.log(np.maximum(probs[0], 1e-15)))
    return loss, probs[0], s_pos

# Vektor Uji Sesuai Seksi 3
q = np.array([0.8, 0.6])
k_pos = np.array([0.7071068, 0.7071068])
k_negs = np.array([[-0.6, 0.8], [-0.8, -0.6]])
tau = 0.1

loss, p_pos, s_pos = infonce_loss(q, k_pos, k_negs, tau=tau)
print(f"Kesamaan Positif: {s_pos:.6f}")
print(f"Posterior Positif P(+): {p_pos:.7f}")
print(f"InfoNCE Loss: {loss:.8f}")

# Assert ground truth analitis
np.testing.assert_allclose(s_pos, 0.9899495, atol=1e-5)
np.testing.assert_allclose(p_pos, 0.9999498, atol=1e-6)
np.testing.assert_allclose(loss, 5.02e-5, atol=1e-6)
print("VALIDASI BERHASIL: Hasil komputasi InfoNCE presisi sesuai derivasi matematis!")

In [ ]:
# 4. Eksperimen Sensitivitas Suhu & Simulasi Zero-Shot CLIP
# Evaluasi pengaruh variasi temperatur tau
print("--- Sweep Temperatur tau ---")
for t in [0.05, 0.10, 0.20, 0.50, 1.00]:
    l, p, _ = infonce_loss(q, k_pos, k_negs, tau=t)
    print(f"tau = {t:4.2f} -> P(+) = {p:.5f}, Loss = {l:.6f}")

print("\n--- Simulasi Zero-Shot Multimodal Retrieval ---")
classes = ["kucing", "anjing", "mobil", "pesawat"]
# Representasi teks prototipe 8-dimensi
np.random.seed(42)
text_bank = l2_norm(np.random.randn(len(classes), 8))

# Kueri citra sintetis yang dirancang berasosiasi dengan kelas 'anjing' (indeks 1)
img_vector = text_bank[1] + 0.1 * np.random.randn(8)
img_vector = l2_norm(img_vector)

# Komputasi kesamaan kosinus
similarities = np.dot(text_bank, img_vector.T)
logits = similarities / 0.07
probs = np.exp(logits - np.max(logits)) / np.sum(np.exp(logits - np.max(logits)))

pred_idx = np.argmax(probs)
print(f"Target Asli: '{classes[1]}' | Prediksi Zero-Shot: '{classes[pred_idx]}' (Keyakinan: {probs[pred_idx]*100:.2f}%)")
assert pred_idx == 1, "Prediksi zero-shot gagal mencocokkan prototipe yang tepat!"
print("VERIFIKASI ZERO-SHOT: Lulus deterministik.")

In [ ]:
# 5. Spesifikasi Eksekusi Model Fondasi (Google Colab / GPU)
if HAS_TORCH:
    print("Mode PyTorch Aktif.")
    q_t = torch.tensor(q, dtype=torch.float32).unsqueeze(0)
    k_t = torch.tensor(k_pos, dtype=torch.float32).unsqueeze(0)
    cos_sim = F.cosine_similarity(q_t, k_t)
    print("PyTorch Cosine Similarity:", cos_sim.item())
else:
    print("PANDUAN EKSEKUSI MODEL FONDASI DI CLOUD / GOOGLE COLAB:")
    print("1. Pasang pustaka OpenCLIP atau Transformers:")
    print("   !pip install open_clip_torch torch torchvision pillow")
    print("2. Muat model ViT-B/32 terlatih:")
    print("   import open_clip")
    print("   model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')")
    print("   tokenizer = open_clip.get_tokenizer('ViT-B-32')")
    print("3. Ekstraksi visual & textual embeddings:")
    print("   image = preprocess(PIL.Image.open('image.jpg')).unsqueeze(0)")
    print("   text = tokenizer(['a photo of a cat', 'a photo of a dog'])")
    print("   image_features = model.encode_image(image)")
    print("   text_features = model.encode_text(text)")
    print("4. Hitung kesamaan kosinus untuk zero-shot inference.")

### 6. Rangkuman & Pembahasan Akademik
1. **Pembelajaran Representasi Mandiri**: Menyingkirkan hambatan label manual dengan merancang pretext tasks cerdas (kontras InfoNCE, masking MAE, dan distilasi DINOv2).
2. **Dinamika Temperatur**: Mengatur sensitivitas margin logit; pemilihan $\tau$ optimal sangat menentukan konvergensi fitur diskriminatif.
3. **Model Fondasi Visual**: Memberikan representasi universal berkemampuan zero-shot transfer yang menjadi pilar kecerdasan visual multimodal masa depan.